# Model Training

In [1]:
import pandas as pd
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

# -----------------------------
# Load Dataset
# -----------------------------
df = pd.read_csv("civic_complaints.csv")

# =============================
# CATEGORY CLASSIFICATION MODEL
# =============================

X_text = df["complaint_text"]
y_category = df["category"]

# Split BEFORE vectorization (no leakage)
X_train_text, X_test_text, y_train_cat, y_test_cat = train_test_split(
    X_text, y_category, test_size=0.2, random_state=42
)

# Improved TF-IDF
vectorizer = TfidfVectorizer(
    ngram_range=(1,2),
    max_features=1000,
    stop_words="english"
)

X_train_tfidf = vectorizer.fit_transform(X_train_text)
X_test_tfidf = vectorizer.transform(X_test_text)

category_model = LogisticRegression(max_iter=1000)
category_model.fit(X_train_tfidf, y_train_cat)

y_pred_cat = category_model.predict(X_test_tfidf)

print("Category Model Accuracy:", accuracy_score(y_test_cat, y_pred_cat))
print("\nCategory Classification Report:\n")
print(classification_report(y_test_cat, y_pred_cat))

# Save category model & vectorizer
joblib.dump(category_model, "complaint_model.pkl")
joblib.dump(vectorizer, "vectorizer.pkl")


# =============================
# PRIORITY PREDICTION MODEL
# =============================

# Encode categorical features
le_category = LabelEncoder()
le_location = LabelEncoder()
le_priority = LabelEncoder()

df["category_encoded"] = le_category.fit_transform(df["category"])
df["location_encoded"] = le_location.fit_transform(df["location_type"])
df["priority_encoded"] = le_priority.fit_transform(df["priority"])

X_priority = df[["category_encoded", "days_pending", "location_encoded"]]
y_priority = df["priority_encoded"]

X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_priority, y_priority, test_size=0.2, random_state=42
)

priority_model = RandomForestClassifier(n_estimators=200, random_state=42)
priority_model.fit(X_train_p, y_train_p)

y_pred_p = priority_model.predict(X_test_p)

print("\nPriority Model Accuracy:", accuracy_score(y_test_p, y_pred_p))
print("\nPriority Classification Report:\n")
print(classification_report(y_test_p, y_pred_p))

# Save priority model & encoders
joblib.dump(priority_model, "priority_model.pkl")
joblib.dump(le_category, "le_category.pkl")
joblib.dump(le_location, "le_location.pkl")
joblib.dump(le_priority, "le_priority.pkl")

print("\nAll Models Trained & Saved Successfully!")

Category Model Accuracy: 0.93

Category Classification Report:

              precision    recall  f1-score   support

    Drainage       0.93      0.92      0.93        75
 Electricity       0.91      0.92      0.91        74
     Garbage       0.96      0.92      0.94        89
 Road Damage       0.94      0.93      0.93        82
 Water Issue       0.91      0.96      0.93        80

    accuracy                           0.93       400
   macro avg       0.93      0.93      0.93       400
weighted avg       0.93      0.93      0.93       400


Priority Model Accuracy: 0.91

Priority Classification Report:

              precision    recall  f1-score   support

           0       1.00      0.88      0.93        89
           1       0.91      0.88      0.89       145
           2       0.86      1.00      0.92        79
           3       0.88      0.92      0.90        87

    accuracy                           0.91       400
   macro avg       0.91      0.92      0.91       400
we

In [2]:
print("Location classes:", le_location.classes_)

Location classes: ['Highway' 'Hospital Area' 'Market' 'Residential' 'School Area']
